# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

## Ranked actions + reason codes

The model is used as a decision-support tool for prioritizing pages that may deserve a content-refresh review.

The queue is ranked by the model's estimated probability that a page belongs to the declining-content class. Higher scores place pages earlier in the review queue.

Reason codes are added so that a person can understand why a page was prioritized. The reason codes are based on observable feature signals rather than claiming that the model knows the cause of a decline.

Example reason codes include:

- LOW_CTR: relatively low click-through rate
- LOW_IMPRESSIONS: relatively low recent impressions
- LOW_ENGAGEMENT: relatively low engagement rate
- OLD_CONTENT: long time since the last update
- LOW_POSITION: relatively weak average search position
- HIGH_WORD_COUNT: unusually high word count compared with the dataset

The queue is intended to help a human decide what to review first. A high model score does not mean that a page definitely needs a refresh.

In [18]:
import pandas as pd
import numpy as np

DATA_PATH = "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print("Dataset shape:", df.shape)
print(df["is_declining_label"].value_counts())

Dataset shape: (30000, 45)
is_declining_label
1    16262
0    13738
Name: count, dtype: int64


In [19]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "days_since_last_update"
]

categorical_features = [
    "content_type",
    "main_intent",
    "competition_level",
    "impression_tier",
    "position_tier"
]

features = numeric_features + categorical_features

X = df[features]
y = df["is_declining_label"]

groups = df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups=groups))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

X_train = train_df[features]
y_train = train_df["is_declining_label"]

X_test = test_df[features]
y_test = test_df["is_declining_label"]

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features)
    ]
)

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)

pipeline.fit(X_train, y_train)

model_probabilities = pipeline.predict_proba(X_test)[:, 1]

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))
print("Training clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())
print("Predictions:", len(model_probabilities))

Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Predictions: 6163


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## 2. Intended use and limits

### Intended use

This model is intended to support content teams, editors, or analysts in prioritizing pages for content-refresh review.

The model ranks pages using the estimated probability that a page belongs to the declining-content class. Pages with higher model scores appear earlier in the review queue.

The output is decision-support, not an automatic decision. A human should review the page and its underlying signals before deciding whether a content refresh is appropriate.

### How the output should be used

1. Review the highest-ranked pages first.
2. Check the model score and reason codes.
3. Inspect the page's actual content and performance signals.
4. Consider other relevant information that may not be included in the model.
5. Make the final refresh decision through human review.

### Limits

The model does not prove that a page is declining because of a specific cause.

A high model score does not mean that a page definitely needs a refresh.

The model was evaluated on the available anonymized dataset using a grouped-by-client split. Its measured performance should not automatically be assumed to be the same on new datasets, time periods, industries, or other populations.

The reason codes describe observable signals associated with the ranking. They should not be interpreted as causal explanations.

The model should therefore be treated as directional decision-support rather than an automated content decision system.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## Human review + the no-go list

Every recommended page should receive human review before an action is taken.

### Human review checklist

Before refreshing a page, the reviewer should check:

- Whether the page is actually relevant to the current business goal.
- Whether the observed traffic and search signals are meaningful.
- Whether the page content is outdated or incomplete.
- Whether the search intent has changed.
- Whether important information is missing.
- Whether the page has technical or indexing issues that could explain the observed signals.
- Whether the recommendation is consistent with current editorial priorities.

### No-go list

The model should not automatically:

- Rewrite or publish content.
- Delete pages.
- Change URLs.
- Change canonical tags.
- Change robots.txt or indexing settings.
- Change internal linking without review.
- Make claims about why Google rankings changed.
- Override an editor's decision.
- Treat a high model score as proof that a page requires a refresh.

The model's role ends at prioritization and decision support. A human remains responsible for the final action.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## Monitoring / retrain triggers

The recommendations may become stale when the data distribution or the relationship between the features and the target changes.

I would review the model when:

- The distribution of important features changes substantially.
- The proportion of declining pages changes substantially.
- Precision@20 decreases meaningfully compared with the current measured result.
- The ranking quality becomes inconsistent across evaluation periods.
- New features or data sources are introduced.
- The definition of the target label changes.
- Search behavior or measurement methods change.
- The model is used on a substantially different population from the data used for evaluation.

A retraining decision should be based on measured evidence rather than a fixed assumption that the model must be retrained on a particular schedule.

The model should also be revalidated using a grouped or time-aware split when appropriate so that evaluation better represents the intended future or unseen-client use case.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [20]:
import pandas as pd
from pathlib import Path

queue = test_df.copy()

queue["model_score"] = model_probabilities

train_medians = {
    "ctr": train_df["ctr"].median(),
    "impressions_90d": train_df["impressions_90d"].median(),
    "engagement_rate": train_df["engagement_rate"].median(),
    "days_since_last_update": train_df["days_since_last_update"].median(),
    "avg_position": train_df["avg_position"].median(),
    "word_count": train_df["word_count"].median()
}

def add_reason_codes(row):
    reasons = []

    if row["ctr"] < train_medians["ctr"]:
        reasons.append("LOW_CTR")

    if row["impressions_90d"] < train_medians["impressions_90d"]:
        reasons.append("LOW_IMPRESSIONS")

    if row["engagement_rate"] < train_medians["engagement_rate"]:
        reasons.append("LOW_ENGAGEMENT")

    if row["days_since_last_update"] > train_medians["days_since_last_update"]:
        reasons.append("OLD_CONTENT")

    if row["avg_position"] > train_medians["avg_position"]:
        reasons.append("LOW_POSITION")

    if row["word_count"] > train_medians["word_count"]:
        reasons.append("HIGH_WORD_COUNT")

    return ", ".join(reasons) if reasons else "NO_MAJOR_SIGNAL"

queue["reason_codes"] = queue.apply(add_reason_codes, axis=1)

ranked_queue = (
    queue
    .sort_values("model_score", ascending=False)
    .reset_index(drop=True)
)

print("Top 20 ranked pages:")

display(
    ranked_queue[
        ["model_score", "reason_codes"]
    ].head(20)
)

output_dir = Path("/content/flyrank-ml-internship/work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

export_path = output_dir / "content_refresh_action_queue.csv"

ranked_queue[
    ["model_score", "reason_codes"]
].head(100).to_csv(
    export_path,
    index=False
)

print(" ")
print("Export created successfully:")
print(export_path)

check = pd.read_csv(export_path)

print("Rows exported:", len(check))
print("Columns:", list(check.columns))

Top 20 ranked pages:


,model_score,reason_codes
0,0.970000,"OLD_CONTENT, LOW_POSITION"
1,0.966667,LOW_IMPRESSIONS
2,0.960000,"LOW_IMPRESSIONS, OLD_CONTENT, LOW_POSITION"
3,0.960000,"LOW_CTR, LOW_IMPRESSIONS, LOW_POSITION"
4,0.953333,"LOW_CTR, LOW_POSITION, HIGH_WORD_COUNT"
5,0.953333,OLD_CONTENT
6,0.943333,"LOW_CTR, OLD_CONTENT"
7,0.943333,LOW_CTR
8,0.940000,NO_MAJOR_SIGNAL
9,0.940000,"LOW_CTR, LOW_IMPRESSIONS"


 
Export created successfully:
/content/flyrank-ml-internship/work/outputs/content_refresh_action_queue.csv
Rows exported: 100
Columns: ['model_score', 'reason_codes']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.